In [30]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from xgboost import XGBClassifier
import pickle

In [31]:

users = pd.read_csv("./Dataset/users.csv")
fusers = pd.read_csv("./Dataset/fusers.csv")

print("Users shape:", users.shape)
print("Fake users shape:", fusers.shape)

users["isFake"] = 0
fusers["isFake"] = 1



Users shape: (3474, 42)
Fake users shape: (3351, 38)


In [32]:
df = pd.concat([users, fusers], ignore_index=True)
df.columns = df.columns.str.strip()
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [33]:
df["account_age_days"] = (
    pd.Timestamp.utcnow()
    - pd.to_datetime(df["created_at"], utc=True)
).dt.days


In [34]:
features = [
    'favourites_count',
    'followers_count',
    'statuses_count',
    'friends_count',
    'default_profile',
    'default_profile_image',
    'profile_use_background_image',
    'utc_offset',
    'listed_count',
    'geo_enabled',
    'lang',
    'account_age_days',
    'isFake'
]

df = df[features]

In [35]:
df["tweets_per_day"] = df["statuses_count"] / (df["account_age_days"] + 1)


In [36]:
df["ff_ratio"] = df["followers_count"] / (df["friends_count"] + 1)

In [37]:

df["lang"] = df["lang"].fillna("unknown")
lang_map = {lang: i for i, lang in enumerate(df["lang"].unique())}
df["lang"] = df["lang"].map(lang_map)

df = df.fillna(0)


In [38]:
X = df.drop("isFake", axis=1)
y = df["isFake"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
base_model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42
)

cv_scores = cross_val_score(base_model, X, y, cv=cv, scoring='f1')
print("CV F1 Mean:", np.mean(cv_scores))



Train shape: (4777, 14)
Test shape: (2048, 14)
CV F1 Mean: 0.9922312052760052


In [39]:
classifier = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    max_depth=7,
    n_estimators=200,
    learning_rate=0.1,
    subsample=1.0,
    colsample_bytree=0.8,
    random_state=42
)

classifier.fit(X_train, y_train)

y_pred = classifier.predict(X_test)

print("\nEvaluation Metrics:")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 7],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

grid = GridSearchCV(
    XGBClassifier(objective='binary:logistic', eval_metric='logloss'),
    param_grid,
    scoring='f1',
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print("\nBest Parameters:", grid.best_params_)

feature_importance = pd.Series(
    best_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("\nTop Features:")
print(feature_importance)




Evaluation Metrics:
Accuracy : 0.994140625
Precision: 0.9950199203187251
Recall   : 0.9930417495029821
F1 Score : 0.9940298507462687

Confusion Matrix:
[[1037    5]
 [   7  999]]
Fitting 3 folds for each of 32 candidates, totalling 96 fits

Best Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.8}

Top Features:
ff_ratio                        0.743155
favourites_count                0.142049
tweets_per_day                  0.027850
account_age_days                0.017300
friends_count                   0.016258
followers_count                 0.010542
lang                            0.008717
geo_enabled                     0.008533
listed_count                    0.008360
statuses_count                  0.006710
utc_offset                      0.004461
default_profile_image           0.003324
default_profile                 0.002740
profile_use_background_image    0.000000
dtype: float32


In [41]:
classifier.feature_names_in_

array(['favourites_count', 'followers_count', 'statuses_count',
       'friends_count', 'default_profile', 'default_profile_image',
       'profile_use_background_image', 'utc_offset', 'listed_count',
       'geo_enabled', 'lang', 'account_age_days', 'tweets_per_day',
       'ff_ratio'], dtype='<U28')

In [42]:
# ===============================
# Save Model
# ===============================
pickle.dump(best_model, open("profile_classifier.pkl", "wb"))
print("\nModel saved as profile_classifier.pkl")

# ===============================
# Single Prediction Example
# ===============================
sample_prob = best_model.predict_proba(X_test.iloc[0].values.reshape(1, -1))
print("\nSample Prediction Probabilities:", sample_prob)



Model saved as profile_classifier.pkl

Sample Prediction Probabilities: [[4.1961670e-05 9.9995804e-01]]
